### 250409 KC解释生成

大概流程：
    （1）对df_KC_tree中的KC按照level分堆，用bert_base_uncased生成嵌入后，在同level内生成topK相似KC的字典（参考gen_prompt中的相关code）
    （2）以KC为中心重构df_Qmat;并按照交互记录df中的频率降序，保留topK个高频pid（且确保相似KC不包含同样的pid）；
    （3）将kc_short中的题目文本内容作为新的列合并至df_Qmat；
    （4）根据规则生成

##### 读取数据

In [1]:
import pandas as pd
import json
import ast
import os
import re
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # 忽略FutureWarning类型的警告
import random
random.seed(42)  # 全局随机种子设置

"""读取数据，统计数量"""
df = pd.read_csv('/mnt/new_pfs/liming_team/auroraX/cdm_data/NIPS/data/train_data/train_task_3_4.csv',
                 usecols=['QuestionId', 'UserId', 'IsCorrect'])
df_Qmat = pd.read_csv('/mnt/new_pfs/liming_team/auroraX/cdm_data/NIPS/data/metadata/question_metadata_task_3_4.csv')
df = df.merge(df_Qmat[['QuestionId', 'SubjectId']], on='QuestionId', how='left')
df_KC_tree = pd.read_csv('/mnt/new_pfs/liming_team/auroraX/cdm_data/NIPS/data/metadata/subject_metadata.csv')
with open(f'data/nips34_short.json', 'r', encoding='utf-8') as file:
    exer_content = json.load(file)

exer_content = {int(k):v for k,v in exer_content.items()}



# 先剔除掉df_KC_tree中没有交互数据的kc
result = set()
for ind, row in df_Qmat.iterrows():        # 3-655，题库涉及的KC（实际操作时将根部的两层KC剔除）
    temp = ast.literal_eval(row['SubjectId'])
    result.update(temp)
print(len(df_KC_tree))
df_KC_tree = df_KC_tree[df_KC_tree['SubjectId'].isin(result)]
print(len(df_KC_tree))


# 根据kc的ID查询level
dict_kid2level = dict(zip(df_KC_tree.loc[:, "SubjectId"], df_KC_tree.loc[:, "Level"]))


# 不同level的KC组成的子df
df_KCbyLevels = {}
for level in [0,1,2,3]:
    # 提取出特定KC的子df
    df_KCbyLevels[level] = df_KC_tree[df_KC_tree['Level'] == level]


# 假设查询id为ind的KC的同级别KC的ID列表
# df_KCbyLevels[dict_kid2level[ind]]["SubjectId"].tolist()


# 获取问题ID计数
n_exer = df['QuestionId'].value_counts(ascending=False)   # 948，降序排列
pids = [pid for pid, _ in n_exer.items()]                 # 0-947，共948题（致密）
pids.sort()
high_freq_pids = n_exer.index.tolist()

388
86


##### 嵌入

In [2]:
import json
import torch
import numpy as np
from transformers import BertTokenizer, BertModel


# NIPS34的KC:将dataframe的两列转为字典，第一列作为键，第二列作为值
questions = dict(zip(df_KC_tree.loc[:, "SubjectId"], df_KC_tree.loc[:, "Name"]))

# 输出文件名
file_emb = r'/mnt/new_pfs/liming_team/auroraX/songchentao/MyCDM/data/nips34_kc_name_embeds.npy'
file_token = r'/mnt/new_pfs/liming_team/auroraX/songchentao/MyCDM/data/nips34_kc_name_tokens.json'
file_simKCs = r'/mnt/new_pfs/liming_team/auroraX/songchentao/MyCDM/data/nips34_kc_sim_top3.json'

# 整理为list
inds = []
contents = []
for ind, kc_name in questions.items():
    contents.append(kc_name)
    inds.append(ind)

# 批量处理文本
tokenizer = BertTokenizer.from_pretrained('/mnt/new_pfs/liming_team/auroraX/songchentao/llama/bert-base-uncased')
model = BertModel.from_pretrained('/mnt/new_pfs/liming_team/auroraX/songchentao/llama/bert-base-uncased')
model.eval()
with torch.no_grad():
    # 分词
    exer_tokenized = tokenizer(contents, padding=True, truncation=True, max_length=256, return_tensors='pt')
    # 嵌入
    bert_output = model(**exer_tokenized)
    exer_emb = bert_output.last_hidden_state[:, 0]  # , :
    exer_emb = torch.nn.functional.normalize(exer_emb, p=2, dim=1)  # 归一化 
    print(exer_emb.shape)

# # 保存嵌入结果
# np.save(file_emb, exer_emb.detach().numpy())

torch.Size([86, 768])


In [3]:
print(torch.sqrt(torch.sum(exer_emb**2, dim=1)))
exer_emb = exer_emb.detach().numpy()

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


##### 获取相似的top3同级kc

In [4]:
# 先按照顺序排好嵌入数组
kc_emb = np.zeros((max(inds)+1, exer_emb.shape[1]))
for idx, ind in enumerate(inds):
    kc_emb[ind, :] = exer_emb[idx, :]


# 计算并保存topk相似kc节点
def calculate_similarity(_embeddings):
    """
    计算学生描述之间的相似度
    """
    _similarity_matrix = np.zeros((len(_embeddings), len(_embeddings)))
    for i in range(len(_embeddings)):
        for j in range(len(_embeddings)):
            # 计算余弦相似度
            similarity = np.dot(_embeddings[i], _embeddings[j])
            _similarity_matrix[i][j] = similarity
    return _similarity_matrix


"""计算并显示相似度矩阵"""
similarity_matrix = calculate_similarity(kc_emb)
# print("\n题目描述相似度矩阵:")
# print(similarity_matrix)

# 转存相似度信息
dict_sim = {}
for ind_row in inds:
    temp = similarity_matrix[ind_row, :].squeeze()
    # 排序并取负号实现倒序
    temp = np.argsort(temp)[::-1].tolist()
    # # 剔除自身
    temp = temp[1:]
    # 剔除不存在的学生id
    temp = [elem for elem in temp if elem in df_KCbyLevels[dict_kid2level[ind_row]]["SubjectId"].tolist()]  # 这里应该是同级KC的list(只在同级KC中搜寻topK相似)
    # 保留top20并记录
    dict_sim[ind_row] = temp[:3]

with open(file_simKCs, 'w', encoding='utf-8') as json_file:
    json.dump(dict_sim, json_file, indent=4, ensure_ascii=False)


In [5]:
print(len(df_KC_tree))
print(df_KC_tree.head())
# print(type(df_KC_tree.loc[1, 'top3_sim']))

86
   SubjectId                           Name  ParentId  Level
0          3                          Maths       NaN      0
1         32                         Number       3.0      1
2         33                         BIDMAS     144.0      3
5         36                       Decimals      32.0      2
6         37  Factors, Multiples and Primes      32.0      2


In [6]:
# 将相似关系、高频题目等生成prompt所需的所有信息都集成到df_KC_tree中
df_KC_tree['top3_sim'] = df_KC_tree['SubjectId'].map(dict_sim)

# 从最高频的题目开始遍历
dict_top5_pids = {k:[] for k,v in dict_sim.items()}
for pid in high_freq_pids:
    # 从df_Qmat获取题目的涉及的kc list
    kcs = ast.literal_eval(df_Qmat.loc[df_Qmat['QuestionId']==pid, "SubjectId"].iloc[0])
    for kc in kcs:
        sim_kcs = df_KC_tree.loc[df_KC_tree['SubjectId']==kc, "top3_sim"].iloc[0]
        if len(dict_top5_pids[kc]) >= 5:
            # 判断dict_top5_pids中对应kc的列表是否已满，是则跳过
            continue
        elif bool(set(sim_kcs) & set(kcs)):
            # 否则判断题目的kcs是否包含当前kc的相似kc，是则跳过
            continue
        else:
            # 否则将题目添加到dict_top5_pids中对应kc的列表中
            dict_top5_pids[kc].append(pid)

# 将高频题目信息都集成到df_KC_tree中
df_KC_tree['top5_exer'] = df_KC_tree['SubjectId'].map(dict_top5_pids)
print(df_KC_tree.head())


   SubjectId                           Name  ParentId  Level      top3_sim  \
0          3                          Maths       NaN      0            []   
1         32                         Number       3.0      1      [49, 71]   
2         33                         BIDMAS     144.0      3  [86, 92, 93]   
5         36                       Decimals      32.0      2  [42, 37, 39]   
6         37  Factors, Multiples and Primes      32.0      2  [40, 36, 38]   

                   top5_exer  
0   [199, 911, 89, 855, 764]  
1  [199, 911, 764, 405, 252]  
2   [19, 290, 881, 912, 749]  
5    [937, 752, 57, 593, 61]  
6   [65, 634, 921, 616, 342]  


##### 生成prompt（目标格式为[{"system": <system_prompt>, "prompt": <user_prompt>}, ...]）
##### 250415 新增英文版prompt

In [8]:
eng = True  # 使用英文prompt

system_prompt_eng = """If you are a seasoned math teacher, you need to generate explanations for each knowledge point in a knowledge graph.
I will provide you with the name of the knowledge point and corresponding example problems, as well as names and example problems of distractor knowledge points that are not equivalent to it. Please carefully compare them and generate a core explanation for each knowledge point. If you do well, I will give you a $10 tip for this round. When making decisions, think as thoroughly as possible; write the thinking part after "Thinking," enclosing the thinking part with the symbols 【】.
Therefore, the description generated for the knowledge point is:
【Thinking: XXX】The core explanation of the knowledge point is: XXX
"""

system_prompt = """假如你是一位资深的数学老师，你需要给知识图谱的每一个知识点生成知识点的解释。
我将给你这个知识点的名称和对应的例题，还有与它不等价的干扰知识点的名称和对应的例题。请仔细比较，生成每一个知识点的核心解释。如果你做的好，这一轮我将给你10美元小费，做决定的时候要尽可能充分思考，思考的部分写在思考后面，思考部分前后用符号【】包裹起来。
所以为该知识点生成的描述是：【思考：XXX】该知识点的核心解释为：XXX
"""

user_prompt = """
知识点名称：{} 
-----
该知识点的例题：###例题1###
题目：{}
###例题2###
题目：{}
###例题3###
题目：{}
###例题4###
题目：{}
###例题5###
题目：{}
-----
干扰知识点：[干扰知识点1]{}
[干扰知识点1的例题]###例题1###
题目：{}
###例题2###
题目：{}
###例题3###
题目：{}
###例题4###
题目：{}
###例题5###
题目：{}
[干扰知识点2]{}
[干扰知识点2的例题]###例题1###
题目：{}
###例题2###
题目：{}
###例题3###
题目：{}
###例题4###
题目：{}
###例题5###
题目：{}
[干扰知识点3]{}
[干扰知识点3的例题]###例题1###
题目：{}
###例题2###
题目：{}
###例题3###
题目：{}
###例题4###
题目：{}
###例题5###
题目：{}
"""

user_prompt_eng = """
Knowledge Point Name: {} 
-----
Example Problems for This Knowledge Point:
### Problem 1 ###
Question: {}
### Problem 2 ###
Question: {}
### Problem 3 ###
Question: {}
### Problem 4 ###
Question: {}
### Problem 5 ###
Question: {}
-----
Distractor Knowledge Points:
[Distractor Knowledge Point 1] {}
[Distractor Knowledge Point 1's Example Problems]
### Problem 1 ###
Question: {}
### Problem 2 ###
Question: {}
### Problem 3 ###
Question: {}
### Problem 4 ###
Question: {}
### Problem 5 ###
Question: {}
[Distractor Knowledge Point 2] {}
[Distractor Knowledge Point 2's Example Problems]
### Problem 1 ###
Question: {}
### Problem 2 ###
Question: {}
### Problem 3 ###
Question: {}
### Problem 4 ###
Question: {}
### Problem 5 ###
Question: {}
[Distractor Knowledge Point 3] {}
[Distractor Knowledge Point 3's Example Problems]
### Problem 1 ###
Question: {}
### Problem 2 ###
Question: {}
### Problem 3 ###
Question: {}
### Problem 4 ###
Question: {}
### Problem 5 ###
Question: {}

"""

list_prompts = []
# 遍历df_KC_tree中的每个KC
for ind, row in df_KC_tree.iterrows():
    temp_contents = []
    idx = row["SubjectId"]
    kc = row["Name"]
    exers = row["top5_exer"]
    sim_kcs = row["top3_sim"]
    # 将当前题目自己的信息汇总保存
    temp_contents.append(kc)
    temp = [exer_content[elem] for elem in exers]
    while len(temp) < 5:
        if eng:
            temp.append("None")
        else:
            temp.append("无")  # 确保五个代表性题目的位置都占上，防止错位
    temp_contents.extend(temp)

    # 遍历相似知识点，整理信息后保存
    for sim_kc in sim_kcs:
        temp_contents.append(df_KC_tree.loc[df_KC_tree["SubjectId"]==sim_kc, "Name"].iloc[0])
        sim_exers = df_KC_tree.loc[df_KC_tree["SubjectId"]==sim_kc, "top5_exer"].iloc[0]
        _temp = [exer_content[elem] for elem in sim_exers]
        while len(_temp) < 5:
            if eng:
                _temp.append("None")
            else:
                _temp.append("无")
        temp_contents.extend(_temp)
    
    # 填补相似KC少于3个的特殊情况
    while len(temp_contents) < 24:
        if eng:
            temp_contents.append("None")
        else:
            temp_contents.append("无")

    # 使用temp_contents填充prompt模板，并汇总记录
    if eng:
        converted_item = {
        "idx": idx,
        "system": system_prompt_eng,
        "prompt": user_prompt_eng.format(*temp_contents)
        }
    else:
        converted_item = {
            "idx": idx,
            "system": system_prompt,
            "prompt": user_prompt.format(*temp_contents)
        }
    list_prompts.append(converted_item)

print(len(list_prompts))
json_string = json.dumps(list_prompts, ensure_ascii=False, indent=4)
print(json_string)

# 将JSON字符串保存到文件
if eng:
    with open(r'/mnt/new_pfs/liming_team/auroraX/songchentao/MyCDM/data/nips34_kc_discription_qwen_prompts_eng.json', 'w', encoding='utf-8') as f:
        f.write(json_string)
else:
    with open(r'/mnt/new_pfs/liming_team/auroraX/songchentao/MyCDM/data/nips34_kc_discription_qwen_prompts.json', 'w', encoding='utf-8') as f:
        f.write(json_string)


86
[
    {
        "idx": 3,
        "system": "If you are a seasoned math teacher, you need to generate explanations for each knowledge point in a knowledge graph.\nI will provide you with the name of the knowledge point and corresponding example problems, as well as names and example problems of distractor knowledge points that are not equivalent to it. Please carefully compare them and generate a core explanation for each knowledge point. If you do well, I will give you a $10 tip for this round. When making decisions, think as thoroughly as possible; write the thinking part after \"Thinking,\" enclosing the thinking part with the symbols 【】.\nTherefore, the description generated for the knowledge point is:\n【Thinking: XXX】The core explanation of the knowledge point is: XXX\n",
        "prompt": "\nKnowledge Point Name: Maths \n-----\nExample Problems for This Knowledge Point:\n### Problem 1 ###\nQuestion: How many floors does John need to go down to get from his flat to the gym?\n##

In [ ]:
print(df_KC_tree.head())
# 无需重复保存
df_KC_tree.to_csv('/mnt/new_pfs/liming_team/auroraX/cdm_data/NIPS/data/metadata/subject_metadata_gathered.csv', index=False)

   SubjectId                           Name  ParentId  Level      top3_sim  \
0          3                          Maths       NaN      0            []   
1         32                         Number       3.0      1      [49, 71]   
2         33                         BIDMAS     144.0      3  [86, 92, 93]   
5         36                       Decimals      32.0      2  [42, 37, 39]   
6         37  Factors, Multiples and Primes      32.0      2  [40, 36, 38]   

                   top5_exer  
0   [199, 911, 89, 855, 764]  
1  [199, 911, 764, 405, 252]  
2   [19, 290, 881, 912, 749]  
5    [937, 752, 57, 593, 61]  
6   [65, 634, 921, 616, 342]  


/AuroraX-00/share_v4/songchentao/anaconda3/envs/DTransformer/lib/python3.10/site-packages/pandas/core/indexes/base.py:7834: RuntimeWarning: invalid value encountered in cast
  values = values.astype(str)


##### 调用qwen API获得返回后，整理格式

In [ ]:
import json

def extract_core_explanation(text):
    marker = "该知识点的核心解释为："
    last_occurrence = text.rfind(marker)
    if last_occurrence != -1:
        return text[last_occurrence + len(marker):]
    return ""

with open(r'data/nips34_kc_discription_qwen_response.json', 'r', encoding='utf-8') as file:
    responses = json.load(file)

refined_contents = []

for temp in responses:
    temp["output"] = extract_core_explanation(temp["output"])
    refined_contents.append(temp)

with open(r'data/nips34_kc_discription_qwen_response_refined.json', 'w', encoding='utf-8') as json_file:
    json.dump(refined_contents, json_file, indent=4, ensure_ascii=False)
